# Add Dimensions to PHyDAP Runoff Data

## Contents
- [Input Datasets for this Notebook](#input-datasets-for-this-notebook)
- [0. Import required modules](#0-import-required-modules)
- [1. Organize data folders and files](#1-organize-data-folders-and-files)
    - [1.1 Create "Runoff" subfolder](#11-create-runoff-subfolder)
    - [1.2 Create subfolders in "Runoff" subfolder ("Raw", "Interim", "Processed")](#12-create-subfolders-in-runoff-subfolder-raw-interim-processed)
    - [1.3 Move input data files to "Raw" subfolder](#13-move-input-data-files-to-raw-subfolder)
- [2. Declare relevant variables (input/output file paths, field names etc.)](#2-declare-relevant-variables-inputoutput-file-paths-field-names-etc)
- [3. Add dimension to PHyDAP raw .nc file](#3-add-dimension-to-phydap-raw-nc-file)
    - [3.1 Extract data from phydap raw file](#31-extract-data-from-phydap-raw-file)
    - [3.2 Specify the basins of interest (option to use HYBAS_ID or indices)](#32-specify-the-basins-of-interest-option-to-use-hybas_id-or-indices)
    - [3.3 Generate new .nc file with latitude dimension](#33-generate-new-nc-file-with-latitude-dimension)
- [4. Generate GridWeights.txt file from PHyDAP shapefile](#4-generate-gridweightstxt-file-from-phydap-shapefile)
    - [4.1 Load data from hydrofabric and PHyDAP shapefiles](#41-load-data-from-hydrofabric-and-phydap-shapefiles)
    - [4.2 Map PHyDAP to hydrofabric subbasin IDs](#42-map-phydap-to-hydrofabric-subbasin-ids)
    - [4.3 Generate and output GridWeights.txt file](#43-generate-and-output-gridweightstxt-file)
- [5. Time offsets](#5-time-offsets)



## Input Datasets for this Notebook

Prior to running this notebook, ```0_depression_depths_calc.ipynb``` must be ran first. Then, the following files must be downloaded and pasted into the "Data" folder at the root project directory:

- ```PHyDAP_era5_hourly_upland_runoff.nc``` from the [FRDR website](https://www.frdr-dfdr.ca/repo/dataset/7ce4bd7a-4bcc-4f8c-8129-32a691f46c8e)
- [PHyDAP Prairie Hydrosheds shapefile](https://www.frdr-dfdr.ca/repo/dataset/7ce4bd7a-4bcc-4f8c-8129-32a691f46c8e) (paste all files inside the zip Version_02 > metadata > PrairieHydroshedsShapefile.zip)



## 0. Import required modules

In [ ]:
# System
import os
from pathlib import Path
import shutil
import sys
# Ensure PWA directory is in the system path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)

# Analysis
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import geopandas as gpd

# Custom Libraries
import src.py.phydap as phydap

# Visualization
import matplotlib.pyplot as plt

In [2]:
data_cat = 'r' #  input("Enter the data category: Runoff (r), snowmelt (s), rainfall(p)")

## 1. Organize data folders and files

### 1.1 Create "Runoff" subfolder

In [15]:
# Set watershed name
WATERSHED_NAME = input("Enter watershed name (e.g., 'grassmere'): ").strip().lower()


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Specify path for watershed folder
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for Runoff folder
if data_cat.lower() == 'r':
    FOLDER_PATH = WATERSHED_PATH + r"/Runoff"
elif data_cat.lower() == 's':
    FOLDER_PATH = WATERSHED_PATH + r"/Snowmelt"
elif data_cat.lower() == 'p':
    FOLDER_PATH = WATERSHED_PATH + r"/Rainfall"
else:
    raise ValueError("Invalid data category. Please enter 'r', 's', or 'p'.")


# Create watershed folder in specified path
os.makedirs(FOLDER_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "Runoff" subfolder ("Raw", "Interim", "Processed")

In [4]:
# Specify paths for depression depths subfolders
NC_RAW_PATH = FOLDER_PATH + r"/Raw/"
NC_INTERIM_PATH = FOLDER_PATH + r"/Interim/"
NC_PROCESSED_PATH = FOLDER_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [NC_RAW_PATH,
                   NC_INTERIM_PATH,
                   NC_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [5]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(NC_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

In [7]:
# Depression depths shapefile name (generated in depression_depths_calc.ipynb)
DEPRESSION_DEPTHS_SHAPEFILE = input('Enter the depression depths shapefile name (without .shp extension): ') + ".shp"  #"CLRH_basins_depression_depths"

# Depression depths shapefile path
DEPRESSION_DEPTHS_PATH = WATERSHED_PATH + r"/DepressionDepths/Processed"


# Copy and paste depression depths files to "Raw" folder
for filename in os.listdir(DEPRESSION_DEPTHS_PATH):
    if filename.startswith(DEPRESSION_DEPTHS_SHAPEFILE):
        source_path = os.path.join(DEPRESSION_DEPTHS_PATH, filename)
        destination_path = os.path.join(NC_RAW_PATH, filename)
        if os.path.isfile(source_path):  # Avoid copying directories
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata

## 2. Declare relevant variables (input/output file paths, field names etc.)

In [8]:
# Variables for the netCDF file
path_out = NC_PROCESSED_PATH
var_units = "mm/h"
dim1_name = "time_hrs"
dim2_name = "HYBAS_ID"

# Output path for the GridWeights.txt file
path_out_grid = NC_PROCESSED_PATH + "GridWeights.txt"

In [10]:
# Function inputs (eventually will become a function) - netCDF
path_in = NC_RAW_PATH + input('Enter the input netCDF file name: ') + ".nc" # + "PHyDAP_RDRS21_hourly_runoff.nc"
var_name = input('Enter the expected variable name: e.g. "cropped_runoff"')

# Input file path for the PHyDAP shapefile
path_in_phydap_shp = NC_RAW_PATH + input('Enter the PHyDAP shapefile name (without .shp extension): ') + ".shp"  #"prairie_hydrosheds_true_WGS84"

# Input file path for the depression depths shapefile
path_in_deps_shp = NC_RAW_PATH + DEPRESSION_DEPTHS_SHAPEFILE

## 3. Add dimension to PHyDAP raw .nc file

### 3.1 Extract data from phydap raw file

In [11]:
# Load phydap data to custom phydap instance
phydap_raw = phydap.phydap_nc(path_in) 

In [ ]:
extracted_data = phydap_raw.extract_to_pd(var_name)

### 3.2 Specify the basins of interest (option to use HYBAS_ID or indices)

In [ ]:
# Specify ids for basins of interest
# basins_of_interest = [293, 308, 317, 321, 323]
# basins_hybas_ids = [7120966470, 7120966730, 7120965000, 7120299860, 7120298981] # Cypress River
basins_hybas_ids = [7120284170, 7120275840, 7120281790, 7120280520, 7120957812, 7120279730, 7120961650, 7120284220]



basins_of_interest = []
for i in basins_hybas_ids:
    basins_of_interest.append((extracted_data["HYBAS_ID"].data == i).nonzero()[0])

basins_of_interest = [int(arr[0]) for arr in basins_of_interest if len(arr) > 0]


# # Construct longitude-like index
# londim_vals = HYBAS_ID[basins_of_interest].astype(np.float64)

In [ ]:
hybas_map = pd.DataFrame({"grid_idx": range(len(lon_dim.data)), "hybas_id": lon_dim.data})
hybas_map.to_csv(NC_PROCESSED_PATH + "hybas_map.csv", index=False)

In [ ]:
# Specify new dimensions (using custom phydap_dim class)

time_dim = phydap.phydap_dim('time', 'hours since 1900-01-01 00:00:00', extracted_data['nt'], 'T',  extracted_data['var_data'])
lon_dim = phydap.phydap_dim('lon', 'degrees_east', len(basins_of_interest), 'X', extracted_data['HYBAS_ID'][basins_of_interest])

In [ ]:
runoff_data = phydap.phydap_var('runoff', var_units, extracted_data['reshaped'][:,:,basins_of_interest])

### 3.3 Generate new .nc file with latitude dimension

In [ ]:
# Define NetCDF output
new_file_name = input('Enter the desired output file name (with .nc extension): ')
output_path = path_out + new_file_name + '.nc'

In [ ]:
lat_dim = phydap.phydap_dim('lat', 'degrees_north', extracted_data['nlat'], 'Y', extracted_data['lat']) # Create false latitude dimension

In [ ]:
phydap.save_pd_to_nc(output_path, dims_list=[time_dim, lat_dim, lon_dim], var_data=runoff_data)

In [ ]:
phydap_raw.nc_close()

## 4. Generate GridWeights.txt file from PHyDAP shapefile

### 4.1 Load data from hydrofabric and PHyDAP shapefiles

In [13]:
# Read the PHyDAP shapefile
phydap_shp = gdp.read_file(path_in_phydap_shp)

# Read the depression depths shapefile
deps_shp = gdp.read_file(path_in_deps_shp)

### 4.2 Map PHyDAP to hydrofabric subbasin IDs

In [14]:
# Create GridWeights.txt file based on where deps hydrofabric centroids fall in the PHyDAP file

# Load grid indices map from PHyDAP processing
hybas_map = pd.read_csv(NC_PROCESSED_PATH + r"hybas_map.csv")

# Filter out invalid geometries
phydap_shp = phydap_shp[phydap_shp.geometry.is_valid & ~phydap_shp.geometry.is_empty].copy()

# Ensure deps hydrofabric is the same CRS as PHyDAP
deps_shp = deps_shp.to_crs(phydap_shp.crs)

# Find centroids for deps hydrofabric subbasins
deps_shp['centroid'] = deps_shp.geometry.centroid

# Load centroids point data into geodataframe
centroids_gdf = gpd.GeoDataFrame(deps_shp[['SubId']], 
                                 geometry=deps_shp['centroid'], 
                                 crs=deps_shp.crs)

# Join subbasin IDs based on centroid location
joined = gpd.sjoin(centroids_gdf, phydap_shp, how='left', predicate='within')
mapping_df = joined[['SubId', 'HYBAS_ID']]


# Write to CSV (not needed but good to have)
mapping_df.to_csv(NC_INTERIM_PATH + "subid_to_phydap_mapping.csv", index=False)


FileNotFoundError: [Errno 2] No such file or directory: 'c:/Users/iyaktubay/PWA/Data/manning/Runoff/Processed/hybas_map.csv'

In [ ]:
# Check if any data points are left blank
missing = mapping_df[mapping_df['HYBAS_ID'].isna()]


# For any centroids that do not fall inside a PHyDAP subbasin, take nearest subbasin
if len(missing) > 0:
    print("Finding nearest PHyDAP subbasin for centroids falling outside of PHyDAP range...")

    # Convert PHyDAP polygons to centroids to speed up distance computation
    phydap_centroids = phydap_shp.copy()
    phydap_centroids['centroid'] = phydap_centroids.geometry.centroid
    phydap_centroids = phydap_centroids.set_geometry('centroid')

    
    # Remove any invalid centroids (unlikely, but just to be safe)
    phydap_centroids = phydap_centroids[phydap_centroids.geometry.notna()].copy()


    # Create list to store additional mappings
    nearest_matches = []


    # Loop through each unmatched point
    for idx, row in centroids_gdf[centroids_gdf['SubId'].isin(missing['SubId'])].iterrows():
        centroid_geom = row.geometry


        # Compute distances to all PHyDAP centroids
        phydap_centroids['distance'] = phydap_centroids.distance(centroid_geom)
        nearest_phydap = phydap_centroids.loc[phydap_centroids['distance'].idxmin()]
        

        # Store the match (SubId, nearest HYBAS_ID)
        nearest_matches.append({'SubId': row['SubId'], 'HYBAS_ID': nearest_phydap['HYBAS_ID']})


    # Convert to DataFrame and append to main mapping_df
    nearest_df = gpd.GeoDataFrame(nearest_matches)
    mapping_df = mapping_df[~mapping_df['HYBAS_ID'].isna()].copy()
    mapping_df = pd.concat([mapping_df, nearest_df], ignore_index=True)
    mapping_df = mapping_df.merge(hybas_map[['hybas_id', 'grid_idx']],
                                  left_on='HYBAS_ID',
                                    right_on='hybas_id',
                                    how='left').drop(columns=['hybas_id'])


    

    print("mapping_df length", len(mapping_df))
    print("nearest_df length", len(nearest_df))


### 4.3 Generate and output GridWeights.txt file

In [ ]:
output_path = NC_PROCESSED_PATH + r"GridWeights.txt"

with open(output_path, 'w') as file:
    file.write(
        f"# ----------------------------------------------\n"
        f"# Raven Input file\n"
        f"# {WATERSHED_NAME}\n"
        f"# ----------------------------------------------\n\n"
        f":GridWeights\n"
        f"   #\n"
        f"   # [# HRUs]\n"
        f"   :NumberHRUs       {len(mapping_df)}\n"
        f"   :NumberGridCells  {phydap_shp.shape[0]}\n"
        f"   #\n"
        f"   # [HRU ID] [Cell #] [w_kl]"
    )
    
    for _, row in mapping_df.iterrows():
        file.write(f"\n  {int(row['SubId']):>10} {int(row['grid_idx']):>10}{1.0:>10.3f}")
    
    file.write("\n:EndGridWeights")

## 5. Time offsets

Alternate workflow for extracting specific PHyDAP datasets (not for standard Raven calibration flow)

In [ ]:
# Create copy of the clipped .nc with new latitude dimension
old_file_name = input('Enter the name of the clipped .nc file (with .nc extension): ')
old_file_path = NC_PROCESSED_PATH + old_file_name
new_file_name = input('Enter the desired output file name (with .nc extension): ')
new_file_path = NC_PROCESSED_PATH + new_file_name

In [ ]:
phydap.time_offset(old_file_path, new_file_path, offset=10)

In [ ]:
# # Testing the function
# rdrs_clipped = phydap.phydap_nc(filepath = new_file_path, var_name='runoff', file_setting='r+')
# print(rdrs_clipped.file.variables['time'][:10])
# rdrs_clipped.nc_close()

### Old Code: Generate and plot timeseries from PHyDAP nc files

In [ ]:

# cypress_runoff = phydap_runoff['reshaped'][:,0,317]
# cypress_rainfall = extract_phydap(input_file=(NC_RAW_PATH + '/PHyDAP_RDRS21_hourly_rainfall.nc'), var_name='rainfall')['reshaped'][:,0,317]
# phydap_snowmelt = extract_phydap(input_file=(NC_RAW_PATH + '/PHyDAP_RDRS21_hourly_snowmelt.nc'), var_name='snowmelt')
# cypress_snowmelt = phydap_snowmelt['reshaped'][:,0,317]
# timestamps = phydap_snowmelt['var_data']
# HYBAS_ID = phydap_snowmelt['HYBAS_ID']

In [ ]:
# Export a subset of the data to csv
# pd.DataFrame({'time_hrs_since_1900': var_data, 'precip': reshaped[:,0,317].data}).to_csv(path_out + '/rainfall_time_series_317.csv', index=False)


In [ ]:
# Plot a sample runoff graph


# cypress_data = pd.DataFrame({'timestamp': ts, 'rainfall': cypress_rainfall, 'snowmelt': cypress_snowmelt, 'runoff': cypress_runoff})

In [ ]:
# Export the data to csv

# cypress_data.to_csv(NC_PROCESSED_PATH + '/cypress_rainfall_snowmelt_runoff_317.csv', index=False)

In [ ]:
# Plot a sample runoff graph

# plot_df = cypress_data[203500:203600]
# plt.plot(plot_df['timestamp'], plot_df['runoff'])
# plt.plot(plot_df['timestamp'], plot_df['rainfall'])
# plt.plot(plot_df['timestamp'], plot_df['snowmelt'])
# plt.xlabel('Time (hours since 1900-01-01 00:00:00)')

In [ ]:
## Second quality check

# print(np.where(subset == max_val))

# reshaped.shape

## 3.2 Visualize timeseries before writing to new file

In [ ]:
ts = phydap_runoff['var_data'] - 60  # Apply a 60 hour offset to time data

In [ ]:
import matplotlib.pyplot as plt

# Suppose you want to plot for basin_id = 123456
basin_id = 308
# idx = np.where(HYBAS_ID == basin_id)[0][0]  # Find index of basin_id

idx = 0 # testing for first subbasin in dataset

# Plot timeseries for that basin (assuming nlat=1)
plt.figure(figsize=(12, 6))
plt.plot(timestamps[220000:221900], subset[220000:221900, 0, idx])
plt.xlabel("Time (hours)")
plt.ylabel("Runoff (mm/h)")
plt.title(f"Runoff timeseries for HYBAS_ID {basin_id}")
plt.tight_layout()
plt.show()

In [ ]:
var_data = phydap_runoff['var_data'] - 60  # Apply a 60 hour offset to time data

In [ ]:

# # Define dimensions
# ncout.createDimension("lon", len(basins_of_interest))
# ncout.createDimension("lat", 1)
# ncout.createDimension("time", phydap_runoff['nt'])


# # Create and assign dimension variables
# lon_var = ncout.createVariable("lon", "f8", ("lon",))
# lon_var.units = "degrees_east"
# lon_var.axis = "X"
# lon_var[:] = londim_vals

# lat_var = ncout.createVariable("lat", "f8", ("lat",))
# lat_var.units = "degrees_north"
# lat_var.axis = "Y"
# lat_var[:] = phydap_runoff['lat'].astype(np.float64)

# time_var = ncout.createVariable("time", "f8", ("time",))
# time_var.units = "hours since 1900-01-01 00:00:00"
# time_var.axis = "T"
# time_var[:] = var_data.astype(np.float64)

In [ ]:
phydap_raw.nc_close()

In [ ]:
# # Define runoff variable
# fillvalue = 1e32
# runoff_var = ncout.createVariable("runoff", "f4", ("time", "lat", "lon"), fill_value=fillvalue)
# runoff_var.long_name = "runoff"
# runoff_var.units = var_units


# # Write runoff data, in chunks if large
# if subset.size < 2**31 - 1:
#     runoff_var[:, :, :] = subset
# else:
#     for period in range(nt // 100000 + 1):
#         start = period * 100000
#         end = min(nt, (period + 1) * 100000)
#         runoff_var[:, :, start:end] = subset[:, :, start:end]


# # Finalize
# ncout.sync()
# ncout.close()
# print(f"New file written to: {output_path}")

## 4. Explore output data to ensure everything looks good (optional)

In [ ]:
# Open the NetCDF file
output_path = "../Data/Cypress_New/Runoff/Processed/lib_test.nc"

ds = Dataset(output_path, mode="r")

print(ds)                        # Print summary
print(ds.variables.keys())      # List of variable names
print(ds.dimensions.keys())     # List of dimensions

# Check variable attributes
print(ds.variables['cropped_runoff'].long_name)
print(ds.variables['cropped_runoff'].units)
print(ds.variables['time'].units)


In [ ]:
runoff = ds.variables['cropped_runoff'][:10000, 0, :]  # shape: (lon, lat, time)
print("Runoff shape:", runoff.shape)

lon = ds.variables['lon'][:]
lat = ds.variables['lat'][:]
time = ds.variables['time'][:]

print("lon count:", len(lon))
print("lat count:", len(lat))
print("time count:", len(time))

In [ ]:
# Overall stats
print("Runoff (mm/h):")
print("  Mean:", np.mean(runoff))
print("  Min:", np.min(runoff))
print("  Max:", np.max(runoff))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(time[:10000], runoff[:,4100])

In [ ]:
# Average over time for each basin
mean_per_basin = np.mean(runoff, axis=0).data.flatten()  # shape: (lon,) if lat = 


# Create DataFrame of HYBAS_ID and mean runoff
df = pd.DataFrame({
    "HYBAS_ID": lon.astype(int),
    "mean_runoff_mm_per_hr": mean_per_basin
})

df.to_csv(path_out + "/mean_runoff_per_basin.csv", index=False)
print("Exported basin-level runoff summary.")


In [ ]:
ds.close()

In [ ]:
(time[10000] % (24*365.25)) / 24

## Testing files